### RAG Pipeline - Data Ingestion to  Vector DB Pipeline


In [2]:
#Reading the PDF and Storing as Document

import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\ana18\AppData\Local\Temp\ipykernel_27216\2768088664.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [3]:
def process_all_pdf(pdf_path):
    all_document = []
    pdf_dir = Path(pdf_path) #Creates a Path Object
    if pdf_dir.exists():
        print("Path Found !")
        print("Directory Path :" , pdf_dir)
    pdf_files = list(pdf_dir.glob("**/*.pdf")) #Creates list of all the files in the path. These are WindowsPath Objects
    print("List of Files : ", pdf_files)
    print(f"Count of Files : Found {len(pdf_files)} pdf files to process")
    for pdf_file in pdf_files:
        print(f"processing {pdf_file.name} pdf file")
        try:
            loader = PyMuPDFLoader(str(pdf_file)) #pdf_file is a WindowsPath Obj
            document = loader.load() # This extracts Text from the loader object initialized. It Returns 1 documentg per page.

            #Add Metadata to Documentg Created Above
            for doc in document:
                doc.metadata['source_file'] = pdf_file.name 
                doc.metadata['file_type'] = 'pdf'
            all_document.extend(document)
            print(f'{pdf_file.name} is processed')

        except Exception as e:
            print(f"Error at file name:{pdf_file.name} : {e}")
    print(f'total documents loaded : {len(all_document)}')
    return all_document

all_pdf_documents = process_all_pdf(r"..\data\pdf_files")




Path Found !
Directory Path : ..\data\pdf_files
List of Files :  [WindowsPath('../data/pdf_files/Arsenal_History_1990_2026.pdf')]
Count of Files : Found 1 pdf files to process
processing Arsenal_History_1990_2026.pdf pdf file
Arsenal_History_1990_2026.pdf is processed
total documents loaded : 3


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-31T10:16:45+00:00', 'source': '..\\data\\pdf_files\\Arsenal_History_1990_2026.pdf', 'file_path': '..\\data\\pdf_files\\Arsenal_History_1990_2026.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-31T10:16:45+00:00', 'trapped': '', 'modDate': "D:20260531101645+00'00'", 'creationDate': "D:20260531101645+00'00'", 'page': 0, 'source_file': 'Arsenal_History_1990_2026.pdf', 'file_type': 'pdf'}, page_content="Arsenal FC: History from 1990 to 2026\nA concise overview of Arsenal Football Club's journey from the George Graham era through Arsene\nWenger's revolution, the Emirates transition years, and the Mikel Arteta rebuild culminating in the\n2025/26 Premier League title.\n1990–1995: George Graham and Defensive Dominance\nArsenal entered the 1990s as one of Eng

In [5]:
#Chunking

def split_documents(documents,chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        separators= ["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    return split_docs


In [6]:
chunks = split_documents(all_pdf_documents)

Split 3 documents into 5 chunks


In [7]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-31T10:16:45+00:00', 'source': '..\\data\\pdf_files\\Arsenal_History_1990_2026.pdf', 'file_path': '..\\data\\pdf_files\\Arsenal_History_1990_2026.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-31T10:16:45+00:00', 'trapped': '', 'modDate': "D:20260531101645+00'00'", 'creationDate': "D:20260531101645+00'00'", 'page': 0, 'source_file': 'Arsenal_History_1990_2026.pdf', 'file_type': 'pdf'}, page_content="Arsenal FC: History from 1990 to 2026\nA concise overview of Arsenal Football Club's journey from the George Graham era through Arsene\nWenger's revolution, the Emirates transition years, and the Mikel Arteta rebuild culminating in the\n2025/26 Premier League title.\n1990–1995: George Graham and Defensive Dominance\nArsenal entered the 1990s as one of Eng

In [8]:
# Embeddings

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = []

for chunk in chunks:
    vector = model.encode(chunk.page_content)
    embeddings.append(vector)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
print(len(embeddings))

5


In [20]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List,Dict,Any,Tuple
class embedding_manager : 
    def __init__(self,model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name 
        self.model = None 
        self.initialize_embedding_model()

    def initialize_embedding_model(self):
        print(f'Embedding Model Initialization Starting : {self.model_name}')  
        self.model =  SentenceTransformer(self.model_name)

    def generate_embeddings(self, texts : List[str]) -> np.ndarray :
        embeddings = self.model.encode(
            texts,show_progress_bar = True
        )

        return embeddings
    
           

In [21]:
embed_man = embedding_manager()

Embedding Model Initialization Starting : all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
# Genearte List  of each page content in a variable as text
texts = [c.page_content for c in chunks]

vector_em = embed_man.generate_embeddings(texts)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [28]:
vector_em.shape

(5, 384)

In [ ]:
#Vector Store
import chromadb 
from chromadb.config import Settings
  